# Dolt measurements and linear fit in Google Colab

Run the cells in order. First replace `YOUR_DOLTHUB_USERNAME` below with the owner of your DoltHub repository. The example assumes the repository is named `test-database` and is public. For a private repository, DoltHub authentication must be configured before cloning.

This notebook installs Dolt, clones the database into the temporary Colab runtime, starts a local Dolt SQL server, and runs the same analysis as `test4.py`. Push your latest changes from your Mac to DoltHub before starting if you want them included.

Colab runtimes are temporary. The clone, server, CSV, and PNG disappear when the runtime is reset. Download the outputs with the final cell, or save them to Google Drive. A new runtime requires running the setup again.

## 1. Install the software

Run once per new Colab runtime. The Dolt installer downloads the current release.

In [ ]:
%pip install -q numpy pandas matplotlib scipy sqlalchemy pymysql

In [ ]:
import subprocess

subprocess.run(
    ["bash", "-c", "curl -fsSL https://github.com/dolthub/dolt/releases/latest/download/install.sh | sudo bash"],
    check=True,
)
subprocess.run(["dolt", "version"], check=True)

## 2. Clone the database

Enter the owner and repository name from your DoltHub URL. If you rerun this cell in the same runtime, it keeps the existing clone; it does not update it. For a fresh copy, restart the runtime and rerun the notebook.

In [ ]:
from pathlib import Path
import subprocess

DOLTHUB_OWNER = "mbradle"  # Change this
DOLTHUB_DATABASE = "test-database"       # Change if needed
DATABASE_DIR = Path("/content/test-database")

if DOLTHUB_OWNER == "YOUR_DOLTHUB_USERNAME" or not DOLTHUB_OWNER.strip():
    raise ValueError("Set DOLTHUB_OWNER to your DoltHub username or organization.")

if DATABASE_DIR.exists():
    if not (DATABASE_DIR / ".dolt").is_dir():
        raise RuntimeError(f"{DATABASE_DIR} exists but is not a Dolt database.")
    print(f"Using existing clone: {DATABASE_DIR}. It has not been updated.")
else:
    subprocess.run(
        ["dolt", "clone", f"{DOLTHUB_OWNER}/{DOLTHUB_DATABASE}", str(DATABASE_DIR)],
        check=True,
    )
    print(f"Cloned database to {DATABASE_DIR}")

## 3. Start the Dolt SQL server

The server runs in the background inside this Colab runtime. The readiness check connects to the database before continuing.

In [ ]:
import time
import pymysql

# Reuse the process if this cell is run again.
if "dolt_server" not in globals() or dolt_server.poll() is not None:
    server_log = open("/content/dolt_server.log", "a")
    dolt_server = subprocess.Popen(
        ["dolt", "sql-server", "--host", "127.0.0.1", "--port", "3306"],
        cwd=str(DATABASE_DIR), stdout=server_log, stderr=subprocess.STDOUT,
    )

for attempt in range(60):
    if dolt_server.poll() is not None:
        raise RuntimeError("Dolt server stopped. See /content/dolt_server.log for details.")
    try:
        connection = pymysql.connect(
            host="127.0.0.1", port=3306, user="root", database="test-database",
            connect_timeout=1,
        )
        connection.close()
        print("Dolt server is ready.")
        break
    except pymysql.MySQLError as exc:
        last_error = exc
        time.sleep(1)
else:
    raise RuntimeError(f"Could not connect to Dolt: {last_error}. See /content/dolt_server.log.")

## 4. Import Python packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import odr
from sqlalchemy import create_engine, text

## 5. Connect to Dolt and read the measurements

Change the connection string if your username, password, port, or database differs.

In [ ]:
engine = create_engine(
    "mysql+pymysql://root@127.0.0.1:3306/test-database"
)

query = text("""
    SELECT x, y, x_error, y_error
    FROM xy_measurements
    ORDER BY x
""")
with engine.connect() as conn:
    df = pd.read_sql(query, conn)
engine.dispose()
print(f"Read {len(df)} measurements from Dolt.")

df.to_csv("xy_measurements.csv", index=False)
print("Saved measurements to xy_measurements.csv")

if len(df) < 3 or df["x"].nunique() < 2:
    raise ValueError(
        "The fit requires at least three measurements "
        "with at least two distinct x values."
    )

df

## 6. Fit a straight line

Orthogonal distance regression uses the one-standard-deviation uncertainties in both x and y.

In [ ]:
def straight_line(parameters, x):
    slope, intercept = parameters
    return slope * x + intercept

data = odr.RealData(
    df["x"].to_numpy(),
    df["y"].to_numpy(),
    sx=df["x_error"].to_numpy(),
    sy=df["y_error"].to_numpy(),
)
model = odr.Model(straight_line)
initial_slope, initial_intercept = np.polyfit(
    df["x"].to_numpy(), df["y"].to_numpy(), deg=1
)
fit = odr.ODR(data, model, beta0=[initial_slope, initial_intercept])
result = fit.run()
if result.info >= 4:
    raise RuntimeError(
        "ODR did not converge successfully: " + "; ".join(result.stopreason)
    )
slope, intercept = result.beta

## 7. Calculate the fitted line and uncertainty envelope

The envelope propagates the parameter covariance and describes uncertainty in the fitted mean.

In [ ]:
covariance = result.cov_beta
slope_error = np.sqrt(covariance[0, 0])
intercept_error = np.sqrt(covariance[1, 1])
x_fit = np.linspace(df["x"].min(), df["x"].max(), 300)
y_fit = slope * x_fit + intercept
fit_variance = (
    covariance[0, 0] * x_fit**2
    + 2.0 * covariance[0, 1] * x_fit
    + covariance[1, 1]
)
fit_error = np.sqrt(np.maximum(fit_variance, 0.0))

## 8. Plot and save the results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(
    df["x"], df["y"], xerr=df["x_error"], yerr=df["y_error"],
    fmt="o", capsize=4, label="Measurements (±1σ)",
)
ax.plot(x_fit, y_fit, color="darkorange", label="ODR linear fit")
ax.fill_between(
    x_fit, y_fit - fit_error, y_fit + fit_error,
    color="darkorange", alpha=0.25, label="Fitted mean ±1σ",
)
fit_text = (
    f"Slope = {slope:.3f} ± {slope_error:.3f}\n"
    f"Intercept = {intercept:.3f} ± {intercept_error:.3f}"
)
ax.text(
    0.05, 0.95, fit_text, transform=ax.transAxes, fontsize=11,
    verticalalignment="top",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
)
ax.set_title("Linear Measurements with Uncertainties in x and y")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig("xy_measurements.png", dpi=150)
plt.show()
print(f"Slope:     {slope:.3f} ± {slope_error:.3f}")
print(f"Intercept: {intercept:.3f} ± {intercept_error:.3f}")

## 9. Download the output files

Run this cell to download the CSV and PNG to your computer.

In [ ]:
from google.colab import files

files.download("xy_measurements.csv")
files.download("xy_measurements.png")